## Setup and Imports

In [1]:
import json
import os
import numpy as np
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import pandas as pd
from tqdm import tqdm
import random

C:\Users\marti\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
###############################
# Device Management Functions #
###############################

def _backend_available(device_type: str) -> bool:
    """Check backend availability, including on older PyTorch builds."""
    if device_type == "cpu":
        return True
    if device_type == "mps":
        backend = getattr(torch.backends, "mps", None)
    elif device_type in ("cuda", "xpu"):
        backend = getattr(torch, device_type, None)
    else:
        return False

    return backend is not None and backend.is_available()


def get_device(preferred=None) -> torch.device:
    """
    Select an available device.

    With no preference, preserve the original priority and add XPU:
        MPS -> CUDA -> XPU -> CPU

    Examples:
        get_device()         # Automatic selection
        get_device("xpu")    # Require an Intel GPU
        get_device("xpu:0")  # Require Intel GPU 0
        get_device("cuda:0")
        get_device("cpu")

    An explicitly requested unavailable device raises an error.
    """
    if preferred is None:
        for device_type in ("mps", "cuda", "xpu", "cpu"):
            if _backend_available(device_type):
                return torch.device(device_type)

    device = torch.device(preferred)

    if device.type not in ("cpu", "cuda", "xpu", "mps"):
        raise ValueError(f"Unsupported device type: {device.type}")

    if not _backend_available(device.type):
        raise RuntimeError(
            f"Requested device '{device}' is unavailable. "
            "Check your PyTorch build, GPU drivers, and hardware."
        )

    if device.type in ("cuda", "xpu") and device.index is not None:
        backend = getattr(torch, device.type)
        device_count = backend.device_count()

        if not 0 <= device.index < device_count:
            raise ValueError(
                f"Invalid device index {device.index}: "
                f"{device_count} {device.type} device(s) available."
            )

    return device


def move_to_device(data, device: torch.device, non_blocking=False):
    """
    Move tensors or models to a device, including nested containers.

    Non-tensor values such as strings, numbers, and None are preserved.
    """
    device = torch.device(device)

    if isinstance(data, torch.Tensor):
        return data.to(device=device, non_blocking=non_blocking)

    if isinstance(data, torch.nn.Module):
        return data.to(device=device, non_blocking=non_blocking)

    if isinstance(data, dict):
        return {
            key: move_to_device(value, device, non_blocking)
            for key, value in data.items()
        }

    if isinstance(data, list):
        return [
            move_to_device(item, device, non_blocking)
            for item in data
        ]

    if isinstance(data, tuple):
        items = [
            move_to_device(item, device, non_blocking)
            for item in data
        ]

        # Preserve named tuples.
        if hasattr(data, "_fields"):
            return type(data)(*items)

        return tuple(items)

    return data


def print_device_info(device: torch.device) -> None:
    """Print information about the selected CPU or GPU."""
    device = get_device(device)
    print(f"Device: {device}")

    if device.type in ("cuda", "xpu"):
        backend = getattr(torch, device.type)
        index = (
            device.index
            if device.index is not None
            else backend.current_device()
        )
        properties = backend.get_device_properties(index)
        gib = 1024 ** 3

        print(f"Backend: {device.type.upper()}")
        print(f"Device index: {index}")
        print(f"GPU name: {backend.get_device_name(index)}")
        print(f"GPU count: {backend.device_count()}")
        print(f"Total memory: {properties.total_memory / gib:.2f} GiB")
        print(
            f"Allocated memory: "
            f"{backend.memory_allocated(index) / gib:.2f} GiB"
        )
        print(
            f"Reserved memory: "
            f"{backend.memory_reserved(index) / gib:.2f} GiB"
        )

    elif device.type == "mps":
        print("MPS backend is active")

        memory_functions = (
            ("Recommended max memory", "recommended_max_memory"),
            ("Current allocated memory", "current_allocated_memory"),
            ("Driver allocated memory", "driver_allocated_memory"),
        )

        for label, function_name in memory_functions:
            function = getattr(torch.mps, function_name, None)

            if callable(function):
                print(f"{label}: {function() / (1024 ** 3):.2f} GiB")

    else:
        print("CPU backend is active")


def maybe_empty_cache(device: torch.device) -> None:
    """
    Release unused GPU allocator cache.

    This does not free memory occupied by live tensors or models.
    """
    device = torch.device(device)

    if not _backend_available(device.type):
        return

    if device.type in ("cuda", "xpu"):
        backend = getattr(torch, device.type)

        # Respect the requested GPU index.
        with backend.device(device):
            backend.empty_cache()

    elif device.type == "mps":
        empty_cache = getattr(torch.mps, "empty_cache", None)

        if callable(empty_cache):
            empty_cache()


#####################
# Seeding Functions #
#####################

def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch random number generators."""
    random.seed(seed)
    np.random.seed(seed)

    # PyTorch seeds its CPU and supported accelerator generators.
    torch.manual_seed(seed)

In [3]:
# Set random seeds for reproducibility
seed_everything(42)

device = get_device()

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"Available device: {device}")

Setup complete
PyTorch version: 2.14.1+xpu
Available device: xpu


In [4]:
# Define entity labels
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique"
]

# Create BIO tags for each entity label
label_list = ['O']  # Outside
for entity_label in ENTITY_LABELS:
    label_list.append(f'B-{entity_label}')  # Beginning
    label_list.append(f'I-{entity_label}')  # Inside

label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for v, k in enumerate(label_list)}

print(f"Total labels: {len(label_list)}")
print(f"\nFirst 10 labels: {label_list[:10]}")

# Model configuration
model_name = "dmis-lab/biobert-v1.1"  # BioBERT for biomedical text
output_model_dir = "models/bert_biomedbert_ner"

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")

Total labels: 27

First 10 labels: ['O', 'B-anatomical location', 'I-anatomical location', 'B-animal', 'I-animal', 'B-bacteria', 'I-bacteria', 'B-biomedical technique', 'I-biomedical technique', 'B-chemical']

Model: dmis-lab/biobert-v1.1
Output directory: models/bert_biomedbert_ner


## Data Loading Functions

In [5]:
def load_ner_data(file_paths):
    """
    Load NER data from multiple JSON files.
    Each file contains documents with entities.
    """
    all_data = {}
    
    for file_path in file_paths:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            all_data.update(data)
            print(f"Loaded {len(data)} documents from {os.path.basename(file_path)}")
        else:
            print(f"Warning: {file_path} not found")
    
    return all_data


def prepare_documents_for_ner(data):
    """
    Convert raw data into structured format for NER.
    Each document has title and abstract as separate text segments.
    """
    documents = []
    
    for pmid, article in data.items():
        # Process title
        title_text = article['metadata']['title']
        title_entities = [e for e in article['entities'] if e['location'] == 'title']
        
        documents.append({
            'pmid': pmid,
            'location': 'title',
            'text': title_text,
            'entities': title_entities
        })
        
        # Process abstract
        abstract_text = article['metadata']['abstract']
        abstract_entities = [e for e in article['entities'] if e['location'] == 'abstract']
        
        documents.append({
            'pmid': pmid,
            'location': 'abstract',
            'text': abstract_text,
            'entities': abstract_entities
        })
    
    return documents


print("Data loading functions defined")

Data loading functions defined


## Load Training and Dev Data

In [11]:
# Load training data from three quality levels
train_files = [
    "../data/Annotations/Train/gold_quality/json_format/train_gold.json",
    "../data/Annotations/Train/silver_quality/json_format/train_silver.json"
]

train_data = load_ner_data(train_files)
train_documents = prepare_documents_for_ner(train_data)

print(f"\nTrain data length: {len(train_data)}")
print(f"Total training documents: {len(train_documents)}")

Loaded 639 documents from train_gold.json
Loaded 811 documents from train_silver.json

Train data length: 1450
Total training documents: 2900


In [12]:
# Load dev data
dev_data = load_ner_data(["../data/Annotations/Dev/json_format/dev.json"])
dev_documents = prepare_documents_for_ner(dev_data)

print(f"\nDev data length: {len(dev_data)}")
print(f"Total dev documents: {len(dev_documents)}")

Loaded 80 documents from dev.json

Dev data length: 80
Total dev documents: 160


In [21]:
# Show example document
example_doc = train_documents[20]
print(f"Example document:")
print(f"  PMID: {example_doc['pmid']}")
print(f"  Location: {example_doc['location']}")
print(f"  Text: {example_doc['text'][:200]}...")
print(f"  Number of entities: {len(example_doc['entities'])}")
print(f"\nFirst 3 entities:")
for entity in example_doc['entities'][:3]:
    print(f"    - '{entity['text_span']}' [{entity['label']}] @ {entity['start_idx']}-{entity['end_idx']}")

Example document:
  PMID: 37175797
  Location: title
  Text: Exploratory Transcriptomic Profiling Reveals the Role of Gut Microbiota in Vascular Dementia....
  Number of entities: 3

First 3 entities:
    - 'Exploratory Transcriptomic Profiling' [biomedical technique] @ 0-35
    - 'Gut Microbiota' [microbiome] @ 57-70
    - 'Vascular Dementia' [DDF] @ 75-91


## Initialize BERT Model and Tokenizer

In [14]:
# Initialize tokenizer and model
print("Initializing BERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)
model.to(device)

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Model loaded with {model.num_labels} labels")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Test tokenization
sample_text = "The gut microbiome plays a role in Parkinson's disease."
tokens = tokenizer.tokenize(sample_text)
print(f"\nSample tokenization: {tokens}")

Initializing BERT tokenizer and model...


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer loaded: BertTokenizerFast
Model loaded with 27 labels
Vocabulary size: 28996

Sample tokenization: ['The', 'gut', 'micro', '##bio', '##me', 'plays', 'a', 'role', 'in', 'Parkinson', "'", 's', 'disease', '.']


## BIO Tag Generation for Training Data

In [15]:
def align_labels_with_tokens(text, entities, tokenizer, label2id):
    """
    Create BIO tags for tokenized text based on character-level entity annotations.
    Uses offset mapping to align character positions with token positions.
    """
    # Tokenize and get offset mapping
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=512
    )
    
    tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
    offset_mapping = encoding['offset_mapping']
    
    # Initialize all labels as 'O' (outside)
    labels = ['O'] * len(tokens)
    
    # Sort entities by start position to handle overlaps
    sorted_entities = sorted(entities, key=lambda e: (e['start_idx'], -(e['end_idx'] - e['start_idx'])))
    
    # Track which tokens have been labeled (to handle overlaps)
    labeled_positions = set()
    
    for entity in sorted_entities:
        entity_start = entity['start_idx']
        entity_end = entity['end_idx']
        entity_label = entity['label']
        
        # Find tokens that overlap with this entity
        entity_token_start = None
        entity_token_end = None
        
        for idx, (token_start, token_end) in enumerate(offset_mapping):
            # Skip special tokens
            if token_start == 0 and token_end == 0:
                continue
            
            # Check if token overlaps with entity
            if token_start < entity_end and token_end > entity_start:
                if entity_token_start is None:
                    entity_token_start = idx
                entity_token_end = idx
        
        # Apply BIO tagging
        if entity_token_start is not None and entity_token_end is not None:
            for idx in range(entity_token_start, entity_token_end + 1):
                # Only label if not already labeled (handle overlaps)
                if idx not in labeled_positions:
                    if idx == entity_token_start:
                        labels[idx] = f'B-{entity_label}'
                    else:
                        labels[idx] = f'I-{entity_label}'
                    labeled_positions.add(idx)
    
    # Convert labels to IDs
    label_ids = [label2id.get(label, label2id['O']) for label in labels]
    
    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': label_ids,
        'tokens': tokens
    }


print("BIO tag generation function defined")

BIO tag generation function defined


## Process Training and Dev Data with BIO Tags

In [16]:
# Process training data
print("Processing training data...")
processed_train = []

for i, doc in enumerate(tqdm(train_documents, desc="Processing train")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_train.append(processed)

print(f"Training data processed: {len(processed_train)} segments")

Processing training data...


Processing train: 100%|██████████| 2900/2900 [00:02<00:00, 1365.24it/s]

Training data processed: 2900 segments


In [18]:
# Process dev data
print("Processing dev data...")
processed_dev = []

for i, doc in enumerate(tqdm(dev_documents, desc="Processing dev")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_dev.append(processed)

print(f"Dev data processed: {len(processed_dev)} segments")

Processing dev data...


Processing dev: 100%|██████████| 160/160 [00:00<00:00, 1119.21it/s]

Dev data processed: 160 segments


In [20]:
# Show example with BIO tags
example_idx = 20
example = processed_train[example_idx]

print(f"Example from training data:")
print(f"  Text: {example['text'][:150]}...")
print(f"  Entities: {len(example['entities'])}")
print(f"\nToken-Label pairs (first 30):")

token_label_pairs = []
for token, label_id in zip(example['tokens'][:30], example['labels'][:30]):
    label = id2label[label_id]
    token_label_pairs.append((token, label))

df = pd.DataFrame(token_label_pairs, columns=['Token', 'Label'])
print(df.to_string(index=False))

Example from training data:
  Text: Exploratory Transcriptomic Profiling Reveals the Role of Gut Microbiota in Vascular Dementia....
  Entities: 3

Token-Label pairs (first 30):
  Token                  Label
  [CLS]                      O
     Ex B-biomedical technique
    ##p I-biomedical technique
 ##lora I-biomedical technique
 ##tory I-biomedical technique
  Trans I-biomedical technique
##cript I-biomedical technique
  ##omi I-biomedical technique
    ##c I-biomedical technique
   Prof I-biomedical technique
##iling I-biomedical technique
    Rev                      O
  ##eal                      O
    ##s                      O
    the                      O
   Role                      O
     of                      O
      G           B-microbiome
   ##ut           I-microbiome
  Micro           I-microbiome
  ##bio           I-microbiome
   ##ta           I-microbiome
     in                      O
      V                  B-DDF
   ##as                  I-DDF
##cular         

## Prepare Dataset for BERT Training

In [22]:
class NERDataset(Dataset):
    """Custom dataset for NER token classification."""
    
    def __init__(self, processed_data, max_length=512):
        self.data = processed_data
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Pad or truncate to max_length
        input_ids = item['input_ids'][:self.max_length]
        attention_mask = item['attention_mask'][:self.max_length]
        labels = item['labels'][:self.max_length]
        
        # Pad if necessary
        padding_length = self.max_length - len(input_ids)
        if padding_length > 0:
            input_ids = input_ids + [tokenizer.pad_token_id] * padding_length
            attention_mask = attention_mask + [0] * padding_length
            labels = labels + [-100] * padding_length  # -100 is ignored by loss
        
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long)
        }


print("Custom dataset class defined")

Custom dataset class defined


In [23]:
# Create datasets
print("Creating training datasets...")

train_dataset = NERDataset(processed_train)
dev_dataset = NERDataset(processed_dev)

print(f"Training dataset: {len(train_dataset)} examples")
print(f"Dev dataset: {len(dev_dataset)} examples")

Creating training datasets...
Training dataset: 2900 examples
Dev dataset: 160 examples


## Configure Training Arguments

In [25]:
# Setup data collator for token classification
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)
print("Data collator initialized")

Data collator initialized


In [26]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=output_model_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

print("Training configuration ready")
print(f"\tBatch size: {training_args.per_device_train_batch_size}")
print(f"\tEpochs: {training_args.num_train_epochs}")
print(f"\tLearning rate: {training_args.learning_rate}")

Training configuration ready
	Batch size: 8
	Epochs: 3
	Learning rate: 2e-05


## Train BERT Model

Note: This cell might take several minutes to hours depending on dataset size and hardware.

**Additional configurations to test:**
- Change hyperparameters (*learning_rate*, *batch_size*, *num_train_epochs*, *weight_decay*)
- Try different pre-trained models (e.g., "allenai/scibert_scivocab_uncased", "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")
- Experiment with max_length for longer contexts

In [27]:
# Initialize Trainer
print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("Trainer initialized")
print(f"\tTraining samples: {len(train_dataset)}")
print(f"\tEvaluation samples: {len(dev_dataset)}")

Initializing Trainer...
Trainer initialized
	Training samples: 2900
	Evaluation samples: 160


C:\Users\marti\AppData\Local\Temp\ipykernel_10004\212542103.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [29]:
# Start training
print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

Starting model training...


Epoch,Training Loss,Validation Loss
1,0.388700,0.287838
2,0.264500,0.265859
3,0.231200,0.258524



TRAINING COMPLETED!
Training time: 31.87 minutes


## Save Trained Model

In [30]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"Model saved to: {output_model_dir}")

Saving trained model...
Model saved to: models/bert_biomedbert_ner


## Load Model for Inference

In [35]:
# Load the trained model for inference
print("Loading trained model for inference...")

inference_model = AutoModelForTokenClassification.from_pretrained(output_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(output_model_dir)
inference_model.to(device)
inference_model.eval()


print(f"Model loaded from: {output_model_dir}")

Loading trained model for inference...
Model loaded from: models/bert_biomedbert_ner


## Inference Function

In [38]:
def predict_entities(model, tokenizer, text, id2label):
    """
    Perform NER inference on a single text.
    Returns list of entities with their positions and labels.
    """
    # Tokenize
    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=512
    )
    
    offset_mapping = encoding.pop('offset_mapping')[0].numpy()
    
    # Move to GPU if available
    if get_device().type != 'cpu':
        encoding = {k: v.to(get_device()) for k, v in encoding.items()}

    # Predict
    with torch.no_grad():
        outputs = model(**encoding)
        predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    
    # Convert predictions to labels
    predicted_labels = [id2label[pred] for pred in predictions]
    
    # Extract entities from BIO tags
    entities = []
    current_entity = None
    
    for idx, (label, (start_char, end_char)) in enumerate(zip(predicted_labels, offset_mapping)):
        # Skip special tokens
        if start_char == 0 and end_char == 0:
            continue
        
        if label.startswith('B-'):
            # Save previous entity if exists
            if current_entity:
                entities.append(current_entity)
            
            # Start new entity
            entity_label = label[2:]  # Remove 'B-' prefix
            current_entity = {
                'start_idx': start_char,
                'end_idx': end_char,
                'label': entity_label,
                'text_span': text[start_char:end_char]
            }
        
        elif label.startswith('I-') and current_entity:
            # Extend current entity
            entity_label = label[2:]  # Remove 'I-' prefix
            if entity_label == current_entity['label']:
                current_entity['end_idx'] = end_char
                current_entity['text_span'] = text[current_entity['start_idx']:end_char]
        
        else:
            # Outside or label mismatch - save current entity
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    # Save last entity if exists
    if current_entity:
        entities.append(current_entity)
    
    return entities


print("Inference function defined")

Inference function defined


## Predict on Dev Set

In [52]:
# Run inference on all dev data
print("Running inference on dev set...")

# Group predictions by document
predictions = {}

for doc in tqdm(dev_documents, desc="Predicting"):
    pmid = doc['pmid']
    location = doc['location']
    text = doc['text']
    
    # Predict entities
    predicted_entities = predict_entities(
        inference_model,
        inference_tokenizer,
        text,
        id2label
    )
    
    # Add location to each entity
    for entity in predicted_entities:
        entity['location'] = location
    
    # Initialize document if not exists
    if pmid not in predictions:
        predictions[pmid] = {'entities': []}

    parsed_predicted_entities = []
    for entity in predicted_entities:
        parsed_predicted_entities.append({
            'start_idx': int(entity['start_idx']),
            'end_idx': int(entity['end_idx'])-1,
            'label': entity['label'],
            'text_span': entity['text_span'],
            'location': entity['location']
        })
    # Add entities to document
    predictions[pmid]['entities'].extend(parsed_predicted_entities)

print(f"Inference completed: {len(predictions)} documents")
total_entities = sum(len(p['entities']) for p in predictions.values())
print(f"Total entities predicted: {total_entities}")

Running inference on dev set...


Predicting: 100%|██████████| 160/160 [00:06<00:00, 25.62it/s]

Inference completed: 80 documents
Total entities predicted: 2551


## Save Predictions

In [53]:
# Save predictions to file
output_path = "predictions/bert_ner_predictions.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"Predictions saved to {output_path}")

Predictions saved to predictions/bert_ner_predictions.json


## Evaluate Performance

In [54]:
# Load evaluation functions from evaluate.py concepts
def remove_duplicated_entities(predictions):
    """Remove duplicated entities from predictions."""
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped
    
    if removed_count > 0:
        print(f"Removed {removed_count} duplicated entities from predictions")

def remove_overlapping_entities_eval(predictions):
    """Remove overlapping entities, keeping longest spans."""
    removed_count = 0

    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]['entities'])
        
        groups = {'title': [], 'abstract': []}
        for ent in predictions[pmid]['entities']:
            loc = ent["location"]
            groups[loc].append(ent)

        keepers = set()
        for loc in groups:
            group = groups[loc]
            group = sorted(group, key=lambda e: e["start_idx"])

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    if ent["start_idx"] < current_end:
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]
            if cluster:
                clusters.append(cluster)

            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length
                keepers.add((longest["start_idx"],
                             longest["end_idx"],
                             longest["location"]))

        deduped = []
        for ent in predictions[pmid]['entities']:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)

        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    if removed_count > 0:
        print(f"Removed {removed_count} overlapping entities")

print("Evaluation helper functions defined")

Evaluation helper functions defined


In [56]:
def evaluate_ner(predictions, ground_truth):
    """Evaluate NER predictions against ground truth."""
    # Remove duplicated and overlapping entities
    remove_duplicated_entities(predictions)
    remove_overlapping_entities_eval(predictions)
    
    LEGAL_ENTITY_LABELS = [
        "anatomical location", "animal", "bacteria", "biomedical technique",
        "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
        "human", "microbiome", "statistical technique"
    ]
    
    ground_truth_NER = dict()
    count_annotated_entities_per_label = {}
    
    for pmid, article in ground_truth.items():
        if pmid not in ground_truth_NER:
            ground_truth_NER[pmid] = []
        for entity in article['entities']:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)
            
            if label not in count_annotated_entities_per_label:
                count_annotated_entities_per_label[label] = 0
            count_annotated_entities_per_label[label] += 1

    count_predicted_entities_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}
    count_true_positives_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}

    for pmid in predictions.keys():
        entities = predictions[pmid]['entities']
        
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            if label not in LEGAL_ENTITY_LABELS:
                continue

            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1

            entry = (start_idx, end_idx, location, text_span, label)
            if pmid in ground_truth_NER and entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label.values())
    count_predicted_entities = sum(count_predicted_entities_per_label.values())
    count_true_positives = sum(count_true_positives_per_label.values())

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall = count_true_positives / (count_annotated_entities + 1e-10)
    micro_f1 = 2 * ((micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10))

    precision, recall, f1 = 0, 0, 0
    n = len(count_annotated_entities_per_label)
    for label in count_annotated_entities_per_label.keys():
        current_precision = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10) 
        current_recall = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label] + 1e-10) 
        
        precision += current_precision
        recall += current_recall
        f1 += 2 * ((current_precision * current_recall) / (current_precision + current_recall + 1e-10))
    
    precision = precision / n
    recall = recall / n
    f1 = f1 / n

    return precision, recall, f1, micro_precision, micro_recall, micro_f1


# Evaluate
precision, recall, f1, micro_precision, micro_recall, micro_f1 = evaluate_ner(predictions, dev_data)

print("="*60)
print("BERT NER BASELINE RESULTS")
print("="*60)
print("\nMacro-averaged Metrics:")
print(f"  Macro-Precision: {precision:.4f}")
print(f"  Macro-Recall:    {recall:.4f}")
print(f"  Macro-F1 Score:  {f1:.4f}")

print("\nMicro-averaged Metrics:")
print(f"  Micro-Precision: {micro_precision:.4f}")
print(f"  Micro-Recall:    {micro_recall:.4f}")
print(f"  Micro-F1 Score:  {micro_f1:.4f}")
print("="*60)

BERT NER BASELINE RESULTS

Macro-averaged Metrics:
  Macro-Precision: 0.6416
  Macro-Recall:    0.6479
  Macro-F1 Score:  0.6327

Micro-averaged Metrics:
  Micro-Precision: 0.6927
  Micro-Recall:    0.7009
  Micro-F1 Score:  0.6968


## Example Predictions

In [57]:
# Show example predictions
print("Example Predictions:\n")

sample_pmids = list(dev_data.keys())[:5]

for pmid in sample_pmids:
    article = dev_data[pmid]
    pred = predictions[pmid]
    
    print(f"Document PMID: {pmid}")
    print(f"Title: {article['metadata']['title'][:100]}...")
    print(f"\nGold entities: {len(article['entities'])}")
    print(f"Predicted entities: {len(pred['entities'])}")
    
    # Show first few predicted entities
    print("\nSample predictions:")
    for entity in pred['entities'][:5]:
        print(f"  - '{entity['text_span']}' [{entity['label']}] in {entity['location']}")
    
    # Calculate match statistics
    gold_set = set()
    for entity in article['entities']:
        gold_set.add((
            entity['start_idx'],
            entity['end_idx'],
            entity['location'],
            entity['text_span'],
            entity['label']
        ))
    
    pred_set = set()
    for entity in pred['entities']:
        pred_set.add((
            entity['start_idx'],
            entity['end_idx'],
            entity['location'],
            entity['text_span'],
            entity['label']
        ))
    
    correct = len(gold_set & pred_set)
    missed = len(gold_set - pred_set)
    wrong = len(pred_set - gold_set)
    
    print(f"\nCorrect: {correct}")
    print(f"Missed: {missed}")
    print(f"Wrong: {wrong}")
    print("-" * 80)
    print()

Example Predictions:

Document PMID: 35766370
Title: Orthopedic Surgery Causes Gut Microbiome Dysbiosis and Intestinal Barrier Dysfunction in Prodromal A...

Gold entities: 45
Predicted entities: 50

Sample predictions:
  - 'Gut Microbiome Dysbiosis' [DDF] in title
  - 'Intestinal Barrier Dysfunction' [DDF] in title
  - 'Prodromal Alzheimer Disease Patients' [human] in title
  - 'gut microbiota' [microbiome] in abstract
  - 'elderly patients' [human] in abstract

Correct: 39
Missed: 6
Wrong: 11
--------------------------------------------------------------------------------

Document PMID: 34912029
Title: Cadmium exposure modulates the gut-liver axis in an Alzheimer's disease mouse model....

Gold entities: 24
Predicted entities: 26

Sample predictions:
  - 'C' [chemical] in title
  - 'human Apolipoprotein' [chemical] in abstract
  - 'ApoE4' [gene] in abstract
  - 'Alzheimer's disease' [DDF] in abstract
  - 'AD' [DDF] in abstract

Correct: 22
Missed: 2
Wrong: 4
------------------------

## Analysis: Entity Distribution by Label

In [58]:
from collections import Counter

# Count entities by label in predictions
pred_label_counts = Counter()
for pmid, pred in predictions.items():
    for entity in pred['entities']:
        pred_label_counts[entity['label']] += 1

# Count entities by label in gold standard
gold_label_counts = Counter()
for pmid, article in dev_data.items():
    for entity in article['entities']:
        gold_label_counts[entity['label']] += 1

print("Entity Distribution by Label:")
print("="*60)
print(f"{'Label':<25} {'Gold':<10} {'Predicted':<10}")
print("-"*60)

all_labels = set(gold_label_counts.keys()) | set(pred_label_counts.keys())
for label in sorted(all_labels):
    print(f"{label:<25} {gold_label_counts[label]:<10} {pred_label_counts[label]:<10}")

print("-"*60)
print(f"{'TOTAL':<25} {sum(gold_label_counts.values()):<10} {sum(pred_label_counts.values()):<10}")

Entity Distribution by Label:
Label                     Gold       Predicted 
------------------------------------------------------------
DDF                       793        784       
anatomical location       169        193       
animal                    152        153       
bacteria                  183        188       
biomedical technique      139        141       
chemical                  366        377       
dietary supplement        64         87        
drug                      75         86        
food                      59         22        
gene                      63         33        
human                     192        188       
microbiome                231        250       
statistical technique     35         49        
------------------------------------------------------------
TOTAL                     2521       2551      
